# PNCP — Identificação de Subenquadramento de Engenharia

TCC MBA IA & Big Data (ICMC/USP) — Profa. Solange O. Rezende

## Metodologia em 3 etapas

1. **Triagem determinística (etapa 0)** — pré-filtro lexical + verificação
   de rito de engenharia (ART/RRT, memorial, NBR…). Casos óbvios são
   reclassificados aqui, antes do ML.
2. **ML para os ambíguos** — TF-IDF + LR/RF/SVC, opcional BERTimbau.
3. **Camadas complementares** — outliers, grafos, CNAE, aditivos.

**Como usar:** rode em ordem. Cada etapa **persiste em disco** (parquet/json).
Se o kernel cair, é só continuar da célula seguinte — nada se perde.

## Célula 1 — Setup (rode 1× por sessão)

In [ ]:
!pip install -q wordcloud tqdm imbalanced-learn nltk mlxtend psutil
!pip install -q pymupdf pdfplumber pytesseract networkx python-louvain openpyxl
!pip install -q sentence-transformers transformers torch statsmodels
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por > /dev/null
print('OK — dependências instaladas')

### 💡 Atualizar o código no meio do pipeline

Se eu fizer commits novos enquanto você está rodando, e você quiser puxar
sem reiniciar tudo, cole isto em qualquer célula:

```python
pncp.atualizar()   # git fetch + reset + descarrega o módulo
import pncp        # reimporta a versão nova
```

Não roda automaticamente em toda célula (seria lento e atrapalharia
ciclos longos como BERTimbau ou coleta), mas é uma linha quando precisar.

## Célula 2 — Drive + clonar o repositório

## Célula 3 — Coleta (com retomada automática)

Endpoint estável `/v1/contratos`, **mês a mês**, com 6 retentativas e
backoff exponencial (4-64s). A cada 10 páginas grava checkpoint
(parquet + JSON de progresso).

**Se o Colab desconectar:**
1. Reabra o notebook (a célula 2 já puxa código atualizado)
2. Rode esta mesma célula novamente — a função detecta os meses já
   completos e a última página parcial e **retoma exatamente do ponto
   onde parou**, sem rebaixar nada.

Para inspecionar o progresso atual sem baixar:
```python
pncp.coleta.status(uf='SP', anos=range(2024, 2027))
```

In [ ]:
# Coleta SP de 2024 a 2026 com retomada automática.
# Pode interromper e rodar de novo — vai continuar do ponto exato.
pncp.coleta.coletar(
    uf='SP',
    anos=range(2024, 2027),
    mes_inicio=1, mes_fim=12,
    max_paginas=200,
    tamanho=500,
)

pncp.ram.monitorar_ram('após coleta')

In [ ]:
# Modo programático (recomendado para reproduzir):
pncp.coleta.coletar(
    uf='SP',
    anos=range(2023, 2026),
    mes_inicio=1, mes_fim=12,
    max_paginas=200,
    tamanho=500,
)

# Alternativa: modo interativo (pergunta os parâmetros no terminal):
# pncp.coleta.coletar_interativo()

pncp.ram.monitorar_ram('após coleta')

### Recuperação (opcional — kernel caiu ou coletas separadas)

Se a sessão caiu no meio da coleta, ou se você baixou ano por ano em
sessões separadas, esta célula recupera/junta o que já está em disco
**sem chamar a API de novo**.

In [ ]:
# Carrega o consolidado (ou checkpoint) já salvo, sem baixar nada:
# df = pncp.coleta.carregar_checkpoint(uf='SP')
# print(f'{len(df):,} contratos prontos')

# Se você fez 1 coleta por ano em sessões separadas, junta tudo:
# df = pncp.coleta.combinar_parquets(uf='SP')

## Célula 4 — EDA + texto + TF-IDF

In [ ]:
pncp.eda.executar()
from pncp import config
caminho = config.caminho(config.SUB_COLETA, 'contratos.parquet')
pncp.texto.preprocessar(caminho)
pncp.texto.marcar_termos_dominio(caminho)
pncp.texto.construir_tfidf(caminho)
pncp.ram.monitorar_ram('após texto/TF-IDF')

## Célula 5 — Triagem inicial (pré-filtro lexical)

Identifica contratos 'geral' que são *obviamente* engenharia pelo objeto.
Sem os PDFs ainda, todos viram `subenquadramento_real` provisório — vão
ser reclassificados na Célula 7 quando tivermos as features de rito.

In [ ]:
pncp.triagem.executar()

## Célula 6 — PDFs (Camada 2): TR/Edital dos óbvios + ambíguos

Prioriza os contratos marcados como `obvio_engenharia` na triagem para
saber se seguiram o rito.

In [ ]:
pncp.pdfs.executar(max_contratos=300)
pncp.ram.monitorar_ram('após PDFs')

## Célula 7 — Triagem refinada (com verificação de rito)

Re-roda a triagem cruzando com features dos PDFs:
- `rotulacao_incorreta_processo_ok` = óbvio + rito seguido (≥2 sinais)
- `subenquadramento_real` = óbvio + rito **não** seguido (violação Lei 14.133)
- `ambiguo` = vai para o ML

In [ ]:
pncp.triagem.executar()

## Célula 8 — Classificação supervisionada (ML para os ambíguos)

In [ ]:
pncp.classificacao.executar(
    fazer_grid=True,
    fazer_holdout=True,
    fazer_mcnemar=True,
    fazer_bootstrap=True,
)
pncp.ram.monitorar_ram('após classificação')

## Célula 9 — Detecção de outliers (anomalias contextuais)

Aplica IsolationForest no cluster 'geral' — contratos atípicos são
candidatos adicionais a subenquadramento.

In [ ]:
pncp.outliers.executar(fazer_iforest=True, fazer_lof=False)

## Célula 10 — Técnicas avançadas (LDA, Label Propagation, Apriori, KMeans)

In [ ]:
pncp.avancado.executar(
    fazer_lda=True,
    fazer_lp=True,
    fazer_apriori=True,
    fazer_kmeans=True,
    fazer_hier=True,
)
pncp.ram.monitorar_ram('após avançado')

## Célula 11 — Embeddings BERTimbau (opcional, mais lento)

Use `tipo='sentence-bert'` para iterar rápido; `'bertimbau'` na versão final.

In [ ]:
# Cada camada num bloco isolado: se uma falhar (rate limit, PDF corrompido,
# rede ruim) as outras continuam.
import traceback

def _tentar(nome, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
        print(f"  ✅ {nome}")
    except Exception as e:
        print(f"  ⚠ {nome} falhou: {type(e).__name__}: {e}")
        traceback.print_exc(limit=2)

_tentar('aditivos', pncp.aditivos.executar, max_contratos=200)
_tentar('grafos',   pncp.grafos.executar)
_tentar('cnae',     pncp.cnae.executar,
        max_consultas=200,
        caminho_excel_crea='/content/drive/MyDrive/PNCP_TCC/cnaes_crea.xlsx')

pncp.ram.monitorar_ram('após complementares')

## Célula 12 — Aditivos + Grafos + CNAE

In [ ]:
pncp.aditivos.executar(max_contratos=200)
pncp.grafos.executar()
pncp.cnae.executar(
    max_consultas=200,
    caminho_excel_crea='/content/drive/MyDrive/PNCP_TCC/cnaes_crea.xlsx',
)
pncp.ram.monitorar_ram('após complementares')

## Célula 13 — Relatório final TCC + consolidação

Combina o veredito da triagem (determinístico, alta confiança) com o ranking ML.

In [ ]:
pncp.relatorio.gerar()
from pathlib import Path
rel = Path(pncp.config.PASTA_DADOS) / pncp.config.SUB_P9 / 'relatorio.md'
print(rel.read_text(encoding='utf-8'))

## Célula 14 — Validação contra ground truth manual (opcional)

1. Abra `dados/cnae/amostra_revisao_manual.csv`
2. Preencha a coluna `revisao_manual`: `subenq` / `ok` / `duv`
3. Rode esta célula

In [ ]:
csv = pncp.config.caminho(pncp.config.SUB_P8, 'amostra_revisao_manual.csv')
pncp.relatorio.validar_ground_truth(csv)

## Glossário

In [ ]:
pncp.relatorio.glossario()